# Validate and compare forced-winning multistrategies

This notebook runs every solver named in `STRATEGY_NAMES`, validates each returned strategy, and compares the strategies pairwise. Change `GAME_PATH` to any PGSolver `.pg` or `.oink` file.

A forced-winning check treats every allowed move as adversarial: if even one permitted resolution can lose, the report contains a concrete prefix-and-cycle counterexample. Exact maximality tries every omitted legal edge inside the winning region for every selected solver, so it may take substantially longer on large games.

In [ ]:
from itertools import combinations
from pathlib import Path
import sys
import time

search_roots = (Path.cwd(), *Path.cwd().parents)
CODE = next(
    (candidate for root in search_roots
     for candidate in (root, root / 'code')
     if (candidate / 'src' / 'parity_bdd_lab').is_dir()),
    None,
)
if CODE is None:
    raise FileNotFoundError(f'Could not locate the code project from {Path.cwd()}')
sys.path.insert(0, str(CODE / 'src'))

import pandas as pd
from IPython.display import display
from parity_bdd_lab import (
    CycleEnumerationLimitError, DEFAULT_METHODS,
    chain_quotient_multistrategy, chain_quotient_multistrategy_stages,
    enumerate_multistrategy_cycles, read_pg, validate_multistrategy,
)

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 200)

# Configuration
# GAME_PATH = CODE / 'data' / 'games' / 'image_example.pg'
GAME_PATH = CODE / 'data' / 'benchmarks' / 'syntcomp2020-pgame' / 'lilydemo03.tlsf.ehoa.oink'
STRATEGY_NAMES = ['Chain-MS', 'CQ-MS', 'CQ-MS-Cached', 'FPJM', 'Zielonka']
CHECK_MAXIMALITY = True       # Applied to every selected strategy
SHOW_ALL_VERTEX_ROWS = True   # False shows only vertices with omitted/addable moves
ENUMERATE_CQ_CYCLES = True
MAX_CQ_CYCLE_VERTICES = 500   # Skip exhaustive cycles above this graph size
MAX_CQ_CYCLES = 10_000        # Guard against exponential cycle output

## Load, generate, and validate every selected strategy

Available names are `DFI`, `DFI-ns`, `FPJM`, `Chain-MS`, `CQ-MS`, `CQ-MS-Cached`, and `Zielonka`. Unknown or duplicate names are rejected before any solver runs. Progress is printed in roughly 5% increments during each exact omitted-edge check. A skipped maximality phase is reported as `unknown`, never as a pass.

In [ ]:
AVAILABLE_STRATEGIES = {**DEFAULT_METHODS, 'CQ-MS': chain_quotient_multistrategy}
unknown_names = [name for name in STRATEGY_NAMES if name not in AVAILABLE_STRATEGIES]
if unknown_names:
    raise ValueError(
        f'Unknown strategy names {unknown_names}; available: {tuple(AVAILABLE_STRATEGIES)}'
    )
if not STRATEGY_NAMES:
    raise ValueError('Select at least one strategy')
if len(STRATEGY_NAMES) != len(set(STRATEGY_NAMES)):
    raise ValueError('STRATEGY_NAMES must not contain duplicates')

game = read_pg(GAME_PATH)
strategy_results = {}
generation_seconds = {}
reports = {}
cq_stages = None

for strategy_name in STRATEGY_NAMES:
    print(f'\n=== {strategy_name} ===')
    started = time.perf_counter()
    if strategy_name == 'CQ-MS':
        cq_stages = chain_quotient_multistrategy_stages(game)
        strategy_results[strategy_name] = cq_stages.maximized
    else:
        strategy_results[strategy_name] = AVAILABLE_STRATEGIES[strategy_name](game)
    generation_seconds[strategy_name] = time.perf_counter() - started

    last_progress = {}
    def show_progress(event, strategy_name=strategy_name):
        step = max(1, event.total // 20)
        if event.completed == 1 or event.completed == event.total or event.completed % step == 0:
            marker = (event.player, event.completed)
            if marker != last_progress.get('marker'):
                # print(
                #     f'{strategy_name}, player {event.player}: checked '
                #     f'{event.completed:,}/{event.total:,} omitted edges '
                #     f'({event.elapsed_seconds:.2f}s)'
                # )
                last_progress['marker'] = marker

    reports[strategy_name] = validate_multistrategy(
        game, strategy_results[strategy_name],
        check_maximality=CHECK_MAXIMALITY,
        progress=show_progress,
    )
    print(
        f'Generated in {generation_seconds[strategy_name]:.4f}s; '
        f'validated in {reports[strategy_name].total_seconds:.4f}s'
    )

## Overall verdict and quality by strategy

`valid` requires correct regions, structural validity, and a forced win for both players. `complete` additionally requires exact inclusion-maximality. Positional solvers can be valid while not complete because they deliberately return only one choice per controlled vertex.

In [ ]:
overall_rows = []
quality_rows = []
for strategy_name, report in reports.items():
    overall_rows.append({
        'strategy': strategy_name,
        'game': report.game_name,
        'vertices': report.vertices,
        'edges': report.edges,
        'regions disjoint': report.regions_disjoint,
        'regions exhaustive': report.regions_exhaustive,
        'regions match reference': report.regions_correct,
        'structurally valid': report.structurally_valid,
        'forced-winning': report.forced_winning,
        'inclusion-maximal': report.inclusion_maximal,
        'valid': report.valid,
        'complete': report.complete,
        'generation seconds': generation_seconds[strategy_name],
        'validation seconds': report.total_seconds,
    })
    for player_report in report.players:
        quality_rows.append({
            'strategy': strategy_name,
            'player': player_report.player,
            'winning vertices': len(player_report.region),
            'controlled vertices': len(player_report.controlled_vertices),
            'allowed edges': player_report.retained_edge_count,
            'possible edges': player_report.possible_edge_count,
            'edge coverage': player_report.allowed_edge_coverage,
            'multi-choice vertices': player_report.multi_choice_vertices,
            'multi-choice coverage': player_report.multi_choice_coverage,
            'positional refinements': player_report.positional_refinements,
            'log10 refinements': player_report.log10_positional_refinements,
            'omitted candidates': len(player_report.maximality_candidates),
            'safely addable': len(player_report.addable_edges),
            'forced-winning': player_report.forced_winning,
            'inclusion-maximal': player_report.inclusion_maximal,
            'maximality seconds': player_report.maximality_seconds,
        })

overall = pd.DataFrame(overall_rows).set_index('strategy')
quality = pd.DataFrame(quality_rows).set_index(['strategy', 'player'])
display(overall)
display(quality)

## Diagnostics and losing counterexamples

Empty tables mean no problem was found. Structural errors prevent the universal-choice solver from running. A losing witness identifies an actual permitted play whose recurring priority gives the opposing player the win.

In [ ]:
region_rows = []
issue_rows = []
witness_rows = []
for strategy_name, report in reports.items():
    for p in report.players:
        for vertex in sorted(p.missing_reference_vertices, key=repr):
            region_rows.append({
                'strategy': strategy_name, 'player': p.player, 'vertex': vertex,
                'issue': 'missing from returned winning region',
            })
        for vertex in sorted(p.unexpected_vertices, key=repr):
            region_rows.append({
                'strategy': strategy_name, 'player': p.player, 'vertex': vertex,
                'issue': 'unexpected in returned winning region',
            })
        issue_groups = {
            'unknown vertex in winning region': p.unknown_region_vertices,
            'malformed edge': p.malformed_edges,
            'not a game edge': p.illegal_edges,
            'source owned by other player': p.wrong_owner_edges,
            'source outside winning region': p.outside_region_sources,
            'target outside winning region': p.escaping_edges,
            'opponent can leave winning region': p.opponent_escape_edges,
            'controlled vertex has no allowed move': p.missing_choice_vertices,
        }
        for issue, values in issue_groups.items():
            for value in sorted(values, key=repr):
                issue_rows.append({
                    'strategy': strategy_name, 'player': p.player,
                    'issue': issue, 'value': value,
                })
        witness = p.counterexample
        if witness is not None:
            witness_rows.append({
                'strategy': strategy_name,
                'strategy player': p.player,
                'start': witness.start,
                'prefix': witness.prefix,
                'cycle': witness.cycle,
                'cycle priorities': witness.cycle_priorities,
                'greatest recurring priority': witness.greatest_recurring_priority,
                'actual winner': witness.winner,
                'losing starts': tuple(sorted(p.losing_starts, key=repr)),
            })

print('Winning-region differences')
display(pd.DataFrame(region_rows, columns=['strategy', 'player', 'vertex', 'issue']))
print('Structural issues')
display(pd.DataFrame(issue_rows, columns=['strategy', 'player', 'issue', 'value']))
print('Losing counterexamples')
display(pd.DataFrame(witness_rows, columns=[
    'strategy', 'strategy player', 'start', 'prefix', 'cycle',
    'cycle priorities', 'greatest recurring priority', 'actual winner', 'losing starts',
]))

## Per-vertex choices and completeness gaps

An addable target is an omitted move that can be enabled while the entire winning region remains forced-winning. Any such row disproves inclusion-maximality for that solver.

In [ ]:
choice_rows = []
addable_rows = []
for strategy_name, report in reports.items():
    for p in report.players:
        for row in p.vertex_choices:
            choice_rows.append({
                'strategy': strategy_name,
                'player': p.player,
                'vertex': row.vertex,
                'priority': row.priority,
                'legal in-region targets': row.legal_targets,
                'allowed targets': row.allowed_targets,
                'omitted targets': row.omitted_targets,
                'safely addable targets': row.addable_targets,
            })
        for source, target in sorted(p.addable_edges, key=repr):
            addable_rows.append({
                'strategy': strategy_name, 'player': p.player,
                'source': source, 'target': target,
            })

choices = pd.DataFrame(choice_rows)
if not SHOW_ALL_VERTEX_ROWS and not choices.empty:
    choices = choices[
        choices['omitted targets'].map(bool) | choices['safely addable targets'].map(bool)
    ]
display(choices)
print('All safely addable omitted edges')
display(pd.DataFrame(
    addable_rows, columns=['strategy', 'player', 'source', 'target']
))

## CQ-MS possible recurrent cycles

The initial and maximized CQ-MS stages are converted to their induced strategy graphs. Each row is one rotation-unique simple cycle permitted by the strategy and all opponent choices. The displayed tuple is closed by repeating its first vertex. These cycles describe possible recurrent behavior; they are not an enumeration of every infinite play.

In [ ]:
cycle_rows = []
cycle_status_rows = []
if not ENUMERATE_CQ_CYCLES:
    cycle_status_rows.append({'status': 'disabled by ENUMERATE_CQ_CYCLES'})
elif cq_stages is None:
    cycle_status_rows.append({'status': 'CQ-MS was not selected'})
elif len(game.vertices) > MAX_CQ_CYCLE_VERTICES:
    cycle_status_rows.append({
        'status': (
            f'skipped: {len(game.vertices):,} vertices exceed '
            f'MAX_CQ_CYCLE_VERTICES={MAX_CQ_CYCLE_VERTICES:,}'
        )
    })
else:
    for stage_name, stage_result in (
        ('initial', cq_stages.initial),
        ('maximized', cq_stages.maximized),
    ):
        for player in (0, 1):
            region = stage_result.even_wins if player == 0 else stage_result.odd_wins
            if not region:
                continue
            try:
                stage_cycles = enumerate_multistrategy_cycles(
                    game, stage_result, player, max_cycles=MAX_CQ_CYCLES
                )
            except CycleEnumerationLimitError as error:
                cycle_status_rows.append({
                    'status': f'{stage_name}, player {player}: {error}'
                })
                continue
            for cycle_number, cycle in enumerate(stage_cycles, 1):
                closed = cycle.vertices + (cycle.vertices[0],)
                cycle_rows.append({
                    'stage': stage_name,
                    'player': player,
                    'cycle': cycle_number,
                    'length': len(cycle.vertices),
                    'closed vertex tuple': closed,
                    'priorities': cycle.priorities,
                    'greatest recurring priority': cycle.greatest_recurring_priority,
                    'winner': cycle.winner,
                    'winning for strategy player': cycle.winner == player,
                })
            cycle_status_rows.append({
                'status': f'{stage_name}, player {player}: {len(stage_cycles):,} cycles'
            })

display(pd.DataFrame(cycle_status_rows, columns=['status']))
display(pd.DataFrame(cycle_rows, columns=[
    'stage', 'player', 'cycle', 'length', 'closed vertex tuple', 'priorities',
    'greatest recurring priority', 'winner', 'winning for strategy player',
]))

## Pairwise strategy comparison

Winning-region agreement compares the solutions. Strategy edges are compared separately per player. Jaccard similarity is the shared-edge count divided by the union; two different edge sets can both be correct.

In [ ]:
comparison_rows = []
for left_name, right_name in combinations(STRATEGY_NAMES, 2):
    left = strategy_results[left_name]
    right = strategy_results[right_name]
    regions_match = (
        left.even_wins == right.even_wins
        and left.odd_wins == right.odd_wins
    )
    for player in (0, 1):
        left_edges = set(left.even_strategy if player == 0 else left.odd_strategy)
        right_edges = set(right.even_strategy if player == 0 else right.odd_strategy)
        shared = left_edges & right_edges
        union = left_edges | right_edges
        comparison_rows.append({
            'left strategy': left_name,
            'right strategy': right_name,
            'player': player,
            'winning regions match': regions_match,
            'left edges': len(left_edges),
            'right edges': len(right_edges),
            'shared edges': len(shared),
            'union edges': len(union),
            'Jaccard similarity': len(shared) / len(union) if union else 1.0,
            'only in left': len(left_edges - right_edges),
            'only in right': len(right_edges - left_edges),
            'identical edge sets': left_edges == right_edges,
        })

display(pd.DataFrame(comparison_rows, columns=[
    'left strategy', 'right strategy', 'player', 'winning regions match',
    'left edges', 'right edges', 'shared edges', 'union edges',
    'Jaccard similarity', 'only in left', 'only in right', 'identical edge sets',
]))